# Add images to the fixed validation set

Auto-label raw images, review them, then add them to `datasets/ready/fixed_val`.

- Images already in `fixed_val` (same hash) are skipped.
- On a filename collision with a different hash, the new image is renamed with a `_v<n>` suffix.
- Only the classes of `ALLOWED_CLASSES` are accepted.
- Previews for manual review go to `datasets/preprocessed/fixed_val_preview`.

In [ ]:
import shutil
import hashlib
import random
import cv2
import torch
from PIL import Image, ImageOps
from tqdm.notebook import tqdm
import numpy as np

from src.detection import GroundingDINODetector
from src.segmentation import SAMSegmenter
from src.pipeline import img_pipeline
from src.data_utils import get_image_hash

In [ ]:
from utils import DisplayPath
Path = DisplayPath

In [ ]:
dirs = [
    PROJECT_DIR := Path("datasets/ready/fixed_val"),
    PREVIEW_DIR := Path("datasets/preprocessed/fixed_val_preview"),
    RAW_SOURCE_DIR := Path("datasets/raw/tom_trashcans"),  # or datasets/raw/IRL_validation_pictures
    NEW_IMAGES_DIR := Path("datasets/new_val/images"),
    NEW_LABELS_DIR := Path("datasets/new_val/labels"),
    TEMP_PIPELINE_DIR := Path("datasets/trashcan_pipeline_output"),
]
for d in dirs:
    d.display()

In [ ]:
# IRL pictures: {"red ball": 0, "human": 1}
# Trashcan pictures have no red ball, don't look for one
ALLOWED_CLASSES = {"human": 1, "trashcan": 2}

In [ ]:
CLASS_IDS = set(ALLOWED_CLASSES.values())

BOX_THRESHOLD = 0.42
TEXT_THRESHOLD = 0.2
PROMPT = " . ".join(ALLOWED_CLASSES.keys())
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

RENAME_ON_NAME_CONFLICT = True
PURGE_OUTPUTS = True
SAMPLES_PER_SPLIT = 20

random.seed(42)
print(f"Device: {DEVICE}")

In [ ]:
if PURGE_OUTPUTS:
    for path in [PREVIEW_DIR, NEW_IMAGES_DIR.parent, TEMP_PIPELINE_DIR]:
        if path.exists():
            shutil.rmtree(path)
            print(f"Removed {path}")
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

## Auto-labeling with GroundingDINO + SAM2

Raw images are labeled into `NEW_IMAGES_DIR` / `NEW_LABELS_DIR`, kept apart from `fixed_val` until reviewed.

In [ ]:
# Loading GroundingDINO is slow, keep the models across re-runs
if 'detector' not in globals():
    detector = GroundingDINODetector(
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        device=DEVICE,
    )
    segmenter = SAMSegmenter()

In [ ]:
# Same pipeline as the training data, so EXIF handling and polygons match
if TEMP_PIPELINE_DIR.exists():
    shutil.rmtree(TEMP_PIPELINE_DIR)

TEMP_DET_DIR = TEMP_PIPELINE_DIR / "detection"
TEMP_SEG_DIR = TEMP_PIPELINE_DIR / "segmentation"
TEMP_LBL_DIR = TEMP_PIPELINE_DIR / "labels"
TEMP_IMG_DIR = TEMP_PIPELINE_DIR / "images"
TEMP_EMPTY_DIR = TEMP_PIPELINE_DIR / "empty"

for d in [TEMP_DET_DIR, TEMP_SEG_DIR, TEMP_LBL_DIR, TEMP_IMG_DIR, TEMP_EMPTY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

raw_files = [p for p in RAW_SOURCE_DIR.rglob("*") if p.is_file() and p.suffix.lower() not in ['.Identifier']]
print(f"Raw files found: {len(raw_files)}")

processed_count = 0
errors = 0

for img_path in tqdm(raw_files, desc="Running Pipeline"):
    try:
        img_pipeline(
            img_path=img_path,
            detect_fn=lambda p: detector.detect(p, text_prompt=PROMPT, return_all_by_label=True),
            segment_fn=segmenter.segment_bbox,
            det_output_dir=TEMP_DET_DIR,
            seg_output_dir=TEMP_SEG_DIR,
            txt_output_dir=TEMP_LBL_DIR,
            empty_dir=TEMP_EMPTY_DIR,
            images_output_dir=TEMP_IMG_DIR
        )
        processed_count += 1
    except Exception as e:
        print(f"Pipeline error on {img_path.name}: {e}")
        errors += 1

print(f"Processed {processed_count}, {errors} errors")

NEW_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
NEW_LABELS_DIR.mkdir(parents=True, exist_ok=True)

# img_pipeline only copies images with at least one detection to TEMP_IMG_DIR
pipeline_images = list(TEMP_IMG_DIR.glob("*"))

In [ ]:
processed_hashes = set()
added_final = 0
skipped_dup = 0

for img_path in tqdm(pipeline_images, desc="Merging labels"):
    # Polygons were computed on the EXIF-rotated image, save that one
    pil_img = ImageOps.exif_transpose(Image.open(img_path))
    if pil_img.mode != 'RGB':
        pil_img = pil_img.convert('RGB')

    hsh = hashlib.md5(pil_img.tobytes()).hexdigest()
    if hsh in processed_hashes:
        skipped_dup += 1
        continue

    # img_pipeline writes TEMP_LBL_DIR/<class name>/<stem>.txt with class id 0,
    # merge them into one file with the real class ids
    final_labels = []
    for class_name, class_id in ALLOWED_CLASSES.items():
        lbl_file = TEMP_LBL_DIR / class_name / f"{img_path.stem}.txt"
        if not lbl_file.exists():
            continue
        for line in lbl_file.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) > 1:
                final_labels.append(f"{class_id} " + " ".join(parts[1:]))

    if not final_labels:
        continue

    pil_img.save(NEW_IMAGES_DIR / f"{hsh}.jpg", quality=95)
    (NEW_LABELS_DIR / f"{hsh}.txt").write_text("\n".join(final_labels))

    processed_hashes.add(hsh)
    added_final += 1

print(f"Added {added_final}, skipped {skipped_dup} duplicates")

In [ ]:
NEW_IMAGES_DIR.display()

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.tiff', '.svg', '.heic'}


def convert_to_jpg(source_folder: Path, output_folder: Path):
    converted = 0
    errors = 0
    for file_path in tqdm(list(source_folder.rglob("*")), desc="Convert to JPG"):
        if ':Zone.Identifier' in str(file_path) or not file_path.is_file():
            continue
        if file_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        out_path = output_folder / file_path.relative_to(source_folder).parent / f"{file_path.stem}.jpg"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            with Image.open(file_path) as img:
                # Same EXIF handling as src/pipeline.py
                img = ImageOps.exif_transpose(img)
                if img.mode in ('RGBA', 'LA', 'P'):
                    bg = Image.new('RGB', img.size, (255, 255, 255))
                    if img.mode == 'P':
                        img = img.convert('RGBA')
                    bg.paste(img, mask=img.split()[-1])
                    img = bg
                elif img.mode != 'RGB':
                    img = img.convert('RGB')
                img.save(out_path, 'JPEG', quality=95)
                converted += 1
        except Exception as e:
            errors += 1
            print(f"Failed on {file_path.name}: {e}")
    print(f"Converted {converted} images to {output_folder}, {errors} errors")
    return output_folder


CONVERT = False

if CONVERT:
    NEW_IMAGES_DIR = convert_to_jpg(NEW_IMAGES_DIR, NEW_IMAGES_DIR.parent / (NEW_IMAGES_DIR.name + "_jpg"))

In [ ]:
(PROJECT_DIR / "images").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "labels").mkdir(parents=True, exist_ok=True)

existing_hashes = {
    get_image_hash(img_path): img_path.name
    for img_path in (PROJECT_DIR / "images").glob("*")
    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']
}
print(f"Existing fixed_val images: {len(existing_hashes)}")

In [ ]:
if not NEW_IMAGES_DIR.exists():
    raise FileNotFoundError(f"NEW_IMAGES_DIR missing: {NEW_IMAGES_DIR}")
if not NEW_LABELS_DIR.exists():
    raise FileNotFoundError(f"NEW_LABELS_DIR missing: {NEW_LABELS_DIR}")

added = 0
skipped_dupe = 0
renamed = 0
errors = 0

for img_path in tqdm(list(NEW_IMAGES_DIR.rglob("*")), desc="Add to fixed_val"):
    if not img_path.is_file() or img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
        continue

    lbl_path = NEW_LABELS_DIR / f"{img_path.stem}.txt"
    if not lbl_path.exists():
        print(f"Missing label for {img_path}")
        errors += 1
        continue

    lines_out = []
    valid = True
    for line in lbl_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 3 or len(parts[1:]) % 2 != 0 or int(float(parts[0])) not in CLASS_IDS:
            valid = False
            break
        lines_out.append(line.strip())
    if not valid:
        print(f"Invalid label format or class: {lbl_path}")
        errors += 1
        continue

    file_hash = get_image_hash(img_path)
    if file_hash in existing_hashes:
        skipped_dupe += 1
        continue

    target_img = PROJECT_DIR / "images" / img_path.name
    if target_img.exists() and RENAME_ON_NAME_CONFLICT:
        suffix = 1
        while (PROJECT_DIR / "images" / f"{img_path.stem}_v{suffix}{img_path.suffix}").exists():
            suffix += 1
        target_img = PROJECT_DIR / "images" / f"{img_path.stem}_v{suffix}{img_path.suffix}"
        renamed += 1
    target_lbl = PROJECT_DIR / "labels" / f"{target_img.stem}.txt"

    shutil.copy(img_path, target_img)
    target_lbl.write_text("".join(line + "\n" for line in lines_out))

    existing_hashes[file_hash] = target_img.name
    added += 1

print(f"Added {added}, renamed {renamed}, skipped {skipped_dupe} duplicates, {errors} errors")

In [ ]:
COLORS = [(0, 0, 255), (0, 255, 0), (255, 0, 0)]  # BGR
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)


def draw_yolo_polygons(img_path: Path, lbl_path: Path, out_path: Path):
    if not img_path.exists():
        return False

    # Read through PIL to apply the EXIF rotation, like the pipeline
    try:
        with Image.open(img_path) as pil_img:
            pil_img = ImageOps.exif_transpose(pil_img)
            if pil_img.mode != 'RGB':
                pil_img = pil_img.convert('RGB')
            img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    except Exception as e:
        print(f"Error reading {img_path}: {e}")
        return False

    h, w = img.shape[:2]
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if len(parts) < 3:
                continue
            cls_id = int(float(parts[0]))
            coords = [float(p) for p in parts[1:]]
            pts = [[int(coords[i] * w), int(coords[i + 1] * h)] for i in range(0, len(coords), 2)]
            if len(pts) < 3:
                continue
            color = COLORS[cls_id % len(COLORS)]
            cv2.polylines(img, [np.array(pts, np.int32).reshape((-1, 1, 2))], True, color, 2)
            cv2.putText(img, f"cls {cls_id}", (pts[0][0], pts[0][1] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    cv2.imwrite(str(out_path), img)
    return True


# Only the new images, fixed_val was already reviewed
imgs = list(NEW_IMAGES_DIR.glob("*.jpg")) + list(NEW_IMAGES_DIR.glob("*.png"))
random.shuffle(imgs)
sel = imgs[:SAMPLES_PER_SPLIT]

for img_path in tqdm(sel):
    draw_yolo_polygons(img_path, NEW_LABELS_DIR / f"{img_path.stem}.txt", PREVIEW_DIR / f"new_{img_path.name}")

PREVIEW_DIR.display()

In [ ]:
# Review: delete the bad previews from PREVIEW_DIR, then run this cell.
# Every fixed_val image without a preview is removed from fixed_val.
CLEAN_UP = False

PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
existing_previews = {p.stem.replace("new_", "") for p in PREVIEW_DIR.glob("*")}

fixed_val_images = {p.stem for p in (PROJECT_DIR / "images").glob("*.jpg")}
fixed_val_images.update(p.stem for p in (PROJECT_DIR / "images").glob("*.png"))

deleted_by_user = fixed_val_images - existing_previews

print(f"Images in fixed_val: {len(fixed_val_images)}")
print(f"Previews: {len(existing_previews)}")
print(f"To remove from fixed_val: {len(deleted_by_user)}")
for img_stem in sorted(deleted_by_user):
    print(f"  - {img_stem}")

if deleted_by_user and CLEAN_UP:
    cleaned = 0
    for img_stem in deleted_by_user:
        for ext in ['.jpg', '.jpeg', '.png']:
            candidate = PROJECT_DIR / "images" / f"{img_stem}{ext}"
            if candidate.exists():
                candidate.unlink()
                cleaned += 1
                break

        lbl_file = PROJECT_DIR / "labels" / f"{img_stem}.txt"
        if lbl_file.exists():
            lbl_file.unlink()

    print(f"Removed {cleaned} images and their labels from fixed_val")